In [27]:
import torch    

In [28]:
class Node:
    def __init__(self, feature_idx = None, value = None, children = None):
        self.value = value #only for leaf node
        
        #internal nodes
        self.feature_idx = feature_idx
        self.children = children if children else  None

In [29]:
class DecisionTree:
    def __init__(self, max_depth = 50,  min_samples_split=2, categorical_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.categorical_features = categorical_features
        self.root = None
    
    def fit(self, X, y):
        self.root = self.build_tree(X, y)

    def most_common_value(self, y):
        return y.mode()[0].item()
    
    def entropy(self, y):
        if (len(y)) == 0:
            return 0.0
        
        _, cnts = y.unique(return_counts = True)
        probs = cnts / len(y)
        return - torch.sum(probs * torch.log2(probs + 1e-9)).item()
    
    def ig_cat(self, X, y, feature_idx, parent_entropy):
        unique_values = X[:, feature_idx].unique()
        total = len(y)
        weighted_child_entropy = 0.0

        for label in unique_values:
            indices = X[:, feature_idx] == label
            child_entropy = self.entropy(y[indices])
            n_child = indices.sum()
            weighted_child_entropy += (n_child / total) * child_entropy

        return parent_entropy - weighted_child_entropy
    
    def ig_num(self, X, y, feature_idx, threshold, parent_entropy):
        left_indices = X[:, feature_idx] <= threshold
        right_indices = ~left_indices

        if left_indices.sum() == 0 or right_indices.sum() == 0:
            return 0.0
        
        n_total = len(y)
        n_left = left_indices.sum()
        n_right = right_indices.sum()

        entropy_left = self.entropy(y[left_indices])
        entropy_right = self.entropy(y[right_indices])

        weighted_child_entropy = (n_left / n_total) * entropy_left + (n_right / n_total) * entropy_right
        return parent_entropy - weighted_child_entropy

    
    def best_split(self, X, y):
        best_gain = -1.0
        best_split = {"feature_idx": None}
        n_features = X.shape[1]
        parent_entropy = self.entropy(y)

        for feature_idx in range(n_features):
            if feature_idx in self.categorical_features:
                gain = self.ig_cat(X, y, feature_idx, parent_entropy)
                if gain > best_gain:
                    best_gain = gain
                    best_split["feature_idx"] = feature_idx
            else:
                thresholds = X[:, feature_idx].unique()
                for threshold in thresholds:
                    gain = self.ig_num(X, y, feature_idx, threshold, parent_entropy)
                    if gain > best_gain:
                        best_gain = gain
                        best_split["feature_idx"] = feature_idx
                        best_split["threshold"] = threshold.item()

        return best_split

    
    def build_tree(self, X, y, depth=0):
        n, m = X.shape
        classes = len(y.unique())


        if (depth >= self.max_depth or classes == 1 or n < self.min_samples_split):
            leaf_value = self.most_common_value(y)
            return Node(value=leaf_value)
        
        children = {}
        best_split_info = self.best_split(X, y)
        feature_idx = best_split_info.get("feature_idx")

        if feature_idx is None:
            leaf_value = self._most_common_label(y)
            print("leaf_value", leaf_value)
            return Node(value=leaf_value)
        
        if feature_idx in self.categorical_features:
            unique_values = X[:, feature_idx].unique()
            for value in unique_values:
                indices = X[:, feature_idx] == value
                children[value.item()] = self.build_tree(X[indices], y[indices], depth + 1)

            return Node(feature_idx=feature_idx, children=children)
        else:
            threshold = best_split_info.get("threshold")
            left_indices = X[:, feature_idx] <= threshold
            right_indices = ~left_indices
            children["left"] = self.build_tree(X[left_indices], y[left_indices], depth + 1)
            children["right"] = self.build_tree(X[right_indices], y[right_indices], depth + 1)
            return Node(feature_idx=feature_idx, value=threshold, children=children)
    
    def get_all_labels(self, node):
        labels = []
        if node.value is not None:
            return [node.value]
        for child_node in node.children.values():
            labels.extend(self.get_all_labels(child_node))
        
        return labels
    
    def predict(self, X):
        return torch.tensor([self.traverse_tree(x, self.root) for x in X])

    def traverse_tree(self, x, node:Node):
        if (node.value is not None and node.children is None):
            return node.value
        
        if (node.feature_idx in self.categorical_features):
            label = x[node.feature_idx].item()
            if label in node.children:
                return self.traverse_tree(x, node.children[label])
            else:
                return self.most_common_value(self.get_all_labels(self.root))
        else:
            if x[node.feature_idx] <= node.value:
                return self.traverse_tree(x, node.children["left"])
            else:
                return self.traverse_tree(x, node.children["right"])





In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

print("Loading Titanic dataset from CSV...")
df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)

# Fill missing 'Age' values with the median
df['Age'].fillna(df['Age'].median(), inplace=True)

# Fill missing 'Embarked' with the mode
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Encode categorical features
le_sex = LabelEncoder()
df['Sex'] = le_sex.fit_transform(df['Sex'])

le_embarked = LabelEncoder()
df['Embarked'] = le_embarked.fit_transform(df['Embarked'])

# Identify categorical features and numerical features
categorical_cols = ['Pclass', 'Sex', 'Embarked']
numerical_cols = ['Age', 'SibSp', 'Parch', 'Fare']
feature_names = categorical_cols + numerical_cols

# Define categorical feature indices
categorical_features_indices = [feature_names.index(col) for col in categorical_cols]

# Separate features and target
X_np = df[feature_names].values
y_np = df['Survived'].values

# Split data
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_np, y_np, test_size=0.2, random_state=42
)

# Standardize numerical features
scaler = StandardScaler()
X_train_np[:, len(categorical_cols):] = scaler.fit_transform(X_train_np[:, len(categorical_cols):])
X_test_np[:, len(categorical_cols):] = scaler.transform(X_test_np[:, len(categorical_cols):])

# Convert to PyTorch tensors
X_train = torch.tensor(X_train_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.long)
X_test = torch.tensor(X_test_np, dtype=torch.float32)
y_test = torch.tensor(y_test_np, dtype=torch.long)

print(f"\nCategorical feature indices: {categorical_features_indices}")


max_depth = 8
min_samples_split = 5

print(f"\nTraining Decision Tree with max_depth={max_depth}, min_samples_split={min_samples_split}...")
tree = DecisionTree(
    max_depth=max_depth, 
    min_samples_split=min_samples_split, 
    categorical_features=categorical_features_indices
)
tree.fit(X_train, y_train)

predictions = tree.predict(X_test)
accuracy = (predictions == y_test).float().mean()
print(f"\nTest Accuracy on Titanic dataset: {accuracy.item() * 100:.2f}%")

print("\nFirst 10 predictions vs. actual labels:")
for i in range(10):
    prediction_label = "Survived" if predictions[i].item() == 1 else "Not Survived"
    actual_label = "Survived" if y_test[i].item() == 1 else "Not Survived"
    print(f"Prediction: {prediction_label}, Actual: {actual_label}")

Loading Titanic dataset from CSV...


C:\Users\d1990\AppData\Local\Temp\ipykernel_19768\44297503.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
C:\Users\d1990\AppData\Local\Temp\ipykernel_19768\44297503.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa


Categorical feature indices: [0, 1, 2]

Training Decision Tree with max_depth=8, min_samples_split=5...

Test Accuracy on Titanic dataset: 78.77%

First 10 predictions vs. actual labels:
Prediction: Not Survived, Actual: Survived
Prediction: Not Survived, Actual: Not Survived
Prediction: Not Survived, Actual: Not Survived
Prediction: Survived, Actual: Survived
Prediction: Survived, Actual: Survived
Prediction: Survived, Actual: Survived
Prediction: Survived, Actual: Survived
Prediction: Not Survived, Actual: Not Survived
Prediction: Survived, Actual: Survived
Prediction: Survived, Actual: Survived
